# U-Net on Lite — Kaggle GPU execution

Runs the repo's `unet` experiment on the free Kaggle GPU. The lite data is already attached to the competition (`/kaggle/input/climx-main-track`). Enable the **GPU** accelerator in the notebook settings before running (right panel → Accelerator → GPU P100/T4).

## 1. Clone the repo + install dependencies (~2 min)

In [ ]:
# Idempotent: clone on first run, pull on re-runs (e.g. to pick up fixes).
if [ -d /kaggle/working/climx ]; then
  cd /kaggle/working/climx && git pull --ff-only
else
  git clone --depth 1 https://github.com/nonewanit/climx-emulator.git /kaggle/working/climx
fi
%cd /kaggle/working/climx
%pip install -q -e .
%pip install -q -r requirements-kaggle.txt
import sys, os
# Belt-and-braces: make the repo importable even if the editable install
# landed in a different environment (!pip vs kernel pip gotcha).
sys.path.insert(0, "/kaggle/working/climx/src")
os.environ["CLIMX_LITE_DIR"] = "/kaggle/input/climx-main-track"   # the attached lite bundle (READ-ONLY)
os.environ["CLIMX_PREPROCESS_DIR"] = "/kaggle/working/data/preprocessing_data_lite"
import climx_project
from climx_project import data, preprocess, paths
from climx_project.paths import ensure_dirs
ensure_dirs()
# Prove the dependency chain actually landed in this kernel:
import hydra, torch, xclim, lightning
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| xclim", xclim.__version__)


## 2. Locate the lite files in the bundle

In [ ]:
# The bundle layout may vary; the check prints what it needs.
try:
    data.check_lite_files()
    print("Lite files found under CLIMX_LITE_DIR")
except Exception as e:
    print("check failed:", e)
    print("--- searching for .nc files under /kaggle/input ---")
    import pathlib
    for p in pathlib.Path("/kaggle/input").rglob("*.nc"):
        print(p)
    raise

## 3. Preprocess (CPU, ~10-20 min — one-time)

In [ ]:
import time
t0 = time.time()
ds = data.open_datasets()
preprocess.run_preprocess_train(ds)
preprocess.run_preprocess_test(ds)
print(f"preprocess done in {(time.time()-t0)/60:.1f} min")

## 4. Train the U-Net (GPU, ~5-20 min)

In [ ]:
!python train.py experiment=unet 2>&1 | tail -30
# smoke check first (optional): add  experiment.trainer.max_epochs=1 experiment.record=false

## 4b. Watch training live (inline TensorBoard)

Kaggle proxies the dashboard into the notebook — curves update while training runs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/runs/tensorboard --bind_all --port 6006


## 5. Results

In [ ]:
from pathlib import Path
print(Path("results/scores.md").read_text())
# Leaderboard submission works directly from this notebook (Kaggle creds are
# built in):  !kaggle competitions submit -c climx-main-track #             -f results/submissions/submission_unet.parquet -m "U-Net lite"